# 1.5 — Ingesta: Enfermedades respiratorias (variable de respuesta)

**Capa 5 del protocolo.** Fuentes candidatas: SINAVE (morbilidad por jurisdicción sanitaria, anual) o datos abiertos SSA/Sonora (posible granularidad municipal/unidad médica).

> **TODO crítico:** verificar la granularidad real disponible. Si solo hay datos anuales por jurisdicción sanitaria, el análisis espacial se hará a ese nivel y el temporal será anual/sub-anual — documentar esta limitación en el reporte.

## Entradas esperadas
- `data/raw/` — tabulados de morbilidad (SINAVE u homólogo estatal).
- `data/interim/` — `fecha`, `jurisdiccion`/`municipio`/`ageb`, `casos_ira`, `casos_asma`, `casos_epoc`, `casos_neumonia`.

## Salidas
- `data/interim/enfermedades_respiratorias.csv`

In [3]:
from pathlib import Path
import sys
import os 
import pandas as pd

ROOT_DIR = Path.cwd().parent.parent  # notebooks/<seccion>/ -> raiz del proyecto
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Definir rutas y parámetros
from sonora_health_climate_eda.config import (
    RAW_DATA_DIR,
    INTERIM_DATA_DIR
)

main_folder = os.path.join(INTERIM_DATA_DIR, "salud")
os.makedirs(main_folder, exist_ok=True)

from sonora_health_climate_eda.schemas import InfeccionesRespiratoriasSonoraSchema # improtar el esquema de validación de datos de sequía

# Casos confirmados de IRA (CIE-10 J00-J06/J20/J21 menos J02.0/J03.0)
# semanal epidemiologica x municipio -> mensual para Hermosillo.
# Fuente local: data/raw/salud/enf_resp_sonora_2010_2020.csv
# (Secretaria de Salud de Sonora, cobertura 2010-2020; ver
#  references/salud/data_dictionary_enf_resp_sonora.md).
# La ventana mensual queda definida por PANEL_START_YEAR/PANEL_END_YEAR.

enfermedades_respiratorias_sonora = pd.read_csv(os.path.join(RAW_DATA_DIR, "salud", "enf_resp_sonora_2010_2020.csv"))



## Validación de la descarga de datos

### Función para validar los datos descargados

In [4]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            df_infecciones_respiratorias = InfeccionesRespiratoriasSonoraSchema(**row.to_dict())
            validated_rows.append(df_infecciones_respiratorias.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [5]:
df_validated = validate_df(enfermedades_respiratorias_sonora)
print(f"Filas válidas: {len(df_validated):,} de {len(enfermedades_respiratorias_sonora):,}")
df_validated.head(5)

Filas válidas: 1,732,444 de 1,732,444


,clave_municipio,municipio,clave_entidad,anio,semana,genero,grupo_edad,clave_enfermedad,clues,casos
0,1,ACONCHI,26,2010,1,Masculino,De 00 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMB000021,1
1,1,ACONCHI,26,2010,1,Masculino,De 05 a 09 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
2,1,ACONCHI,26,2010,1,Femenino,De 15 a 19 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMB000021,1
3,1,ACONCHI,26,2010,1,Masculino,De 50 a 59 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
4,1,ACONCHI,26,2010,1,Femenino,De 50 a 59 años,"J00-J06, J20, J21 EXCEPTO J02.0 Y J03.0",SRIMS000010,1
